In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import scipy.sparse as sp
import yaml
import time
import gget

import anndata as an
import scanpy as sc
import rapids_singlecell as rsc

import cupy as cp
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator

from cuml.manifold.umap import simplicial_set_embedding
from scanpy.tools._utils import get_init_pos_from_paga 
from cuml.manifold.umap import find_ab_params

# Enable `managed_memory`
rmm.reinitialize(
    managed_memory=True,
    pool_allocator=False,
)
cp.cuda.set_allocator(rmm_cupy_allocator)

sc.settings.verbosity = 3

In [2]:
%%time
fpath = "../../resources/gene_names.tsv.gz"
gdf = pd.read_csv(fpath, sep='\t', low_memory=False)
print(f"{gdf.shape=}")

protein_coding = gdf[gdf['Gene type'] == 'protein_coding']['Gene name'].unique()
print(f"N protein coding: {protein_coding.shape=}")

gdf.shape=(73466, 8)
N protein coding: protein_coding.shape=(18149,)
CPU times: user 131 ms, sys: 16.8 ms, total: 148 ms
Wall time: 147 ms


In [3]:
%%time
# Load scenic
fpath = "/nfs/turbo/umms-indikar/shared/projects/DGC/data/processed_data/10kbp_up_10kbp_down_B.csv"
df = pd.read_csv(fpath)
df = df.rename(columns={'Unnamed: 0' : 'gene_name'})
df = df.set_index('gene_name')
print(f"{df.shape=}")
df.head()

df.shape=(27090, 1605)
CPU times: user 3.68 s, sys: 539 ms, total: 4.22 s
Wall time: 4.23 s


,ABL1,ACAA1,ADNP,ADNP2,AEBP2,AFF4,AHCTF1,AHDC1,AHR,AHRR,...,ZSCAN32,ZSCAN4,ZSCAN5A,ZSCAN5B,ZSCAN5C,ZSCAN9,ZXDA,ZXDB,ZXDC,ZZZ3
gene_name,,,,,,,,,,,,,,,,,,,,,
A1BG,3.41,0.0,6.140,2.040,0.0,13.70,20.40,4.890000,2.696000,0.0,...,0.00,1.545,0.0,0.0869,4.64,5.5700,2.0215,2.857667,0.1015,3.54
A1BG-AS1,2.23,0.0,5.610,2.040,0.0,13.30,7.66,3.288750,0.964200,0.0,...,0.00,4.175,0.0,0.0000,1.57,1.7870,1.9600,1.640000,0.0000,1.50
A1CF,4.90,0.0,5.730,3.910,0.0,9.50,11.10,4.343750,3.388000,0.0,...,1.33,1.547,0.0,0.1720,5.30,5.4050,5.5950,4.703333,1.7500,4.74
A2M,4.78,0.0,6.880,0.564,0.0,6.99,6.32,4.982500,2.490000,0.0,...,0.00,0.000,0.0,0.0000,4.51,2.2025,5.0650,5.513334,0.3500,5.20
A2M-AS1,2.86,0.0,0.499,0.266,0.0,5.23,4.57,3.617375,0.762922,0.0,...,0.00,0.000,0.0,1.6800,1.88,1.3700,1.6400,1.406333,0.0000,4.21


In [13]:
%%time
# load HWG
fpath = "/scratch/indikar_root/indikar1/shared_data/HWG/operations/filtered_gene_metadata.csv"
hdf = pd.read_csv(fpath)
print(f"{hdf.shape=}")
hdf.head()

hdf.shape=(18581, 2)
CPU times: user 6.69 ms, sys: 1.78 ms, total: 8.47 ms
Wall time: 19.5 ms


,GeneStableID,ChromosomescaffoldName
0,ENSG00000000419,20
1,ENSG00000000457,1
2,ENSG00000000460,1
3,ENSG00000000938,1
4,ENSG00000000971,1


In [ ]:
break

In [22]:
gdf[gdf['in_both']]['Gene type'].value_counts()

Gene type
protein_coding                    15930
transcribed_unitary_pseudogene        1
Name: count, dtype: int64

In [24]:
15930 / 21774

0.7316065031689171

In [23]:
gdf['Gene type'].value_counts()

Gene type
lncRNA                                26917
protein_coding                        21774
processed_pseudogene                   9678
unprocessed_pseudogene                 2731
misc_RNA                               2256
snRNA                                  1958
miRNA                                  1834
transcribed_unprocessed_pseudogene     1489
TEC                                    1039
snoRNA                                  980
transcribed_processed_pseudogene        939
rRNA_pseudogene                         482
IG_V_pseudogene                         295
IG_V_gene                               225
TR_V_gene                               160
transcribed_unitary_pseudogene          140
TR_J_gene                                93
unitary_pseudogene                       83
rRNA                                     67
IG_D_gene                                64
scaRNA                                   48
TR_V_pseudogene                          46
IG_J_gene             